<a href="https://colab.research.google.com/github/muzazr/favera-model/blob/main/01_momtsim_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q \
    polars \
    pyarrow \
    pandas \
    numpy \
    scikit-learn \
    lightgbm \
    shap \
    matplotlib \
    joblib

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/favera/model/transaction-anomaly"
)

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
import random
import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    precision_score,
    recall_score,
)
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
DATA_PATH = RAW_DATA_DIR / "momtsim_v1.csv"

lazy_df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    ignore_errors=True
)

print(lazy_df.collect_schema())

sample_df = lazy_df.head(5).collect()
sample_df

row_count = lazy_df.select(pl.len()).collect().item()
print(f"jumlah transaksi: {row_count:,}")

Schema({'step': Int64, 'transactionType': String, 'amount': Float64, 'initiator': Int64, 'oldBalInitiator': Float64, 'newBalInitiator': Float64, 'recipient': String, 'oldBalRecipient': Float64, 'newBalRecipient': Float64, 'isFraud': Int64})
jumlah transaksi: 1,720,181


## 2. Memahami Struktur Dataset

In [ ]:
from IPython.display import display, Markdown
import polars as pl

schema = lazy_df.collect_schema()
columns = schema.names()

schema_df = pl.DataFrame({
    "column": columns,
    "data_type": [str(schema[col]) for col in columns]
})

display(Markdown(
    f"Dataset memiliki **{row_count:,} baris** dan "
    f"**{len(columns)} kolom**."
))

display(schema_df)

Dataset memiliki **1,720,181 baris** dan **10 kolom**.

column,data_type
str,str
"""step""","""Int64"""
"""transactionType""","""String"""
"""amount""","""Float64"""
"""initiator""","""Int64"""
"""oldBalInitiator""","""Float64"""
"""newBalInitiator""","""Float64"""
"""recipient""","""String"""
"""oldBalRecipient""","""Float64"""
"""newBalRecipient""","""Float64"""


In [ ]:
preview_df = lazy_df.head(10).collect()

display(preview_df)

EDA_SAMPLE_SIZE = 100_000

eda_df = lazy_df.head(EDA_SAMPLE_SIZE).collect(
    engine="streaming"
)

display(Markdown(
    f"Sampel EDA menggunakan **{eda_df.height:,} baris**."
))

step,transactionType,amount,initiator,oldBalInitiator,newBalInitiator,recipient,oldBalRecipient,newBalRecipient,isFraud
i64,str,f64,i64,f64,f64,str,f64,f64,i64
0,"""TRANSFER""",19824.96,4537027967639631,187712.18,167887.22,"""4875702729424478""",8.31,19833.27,1
0,"""PAYMENT""",598.97,4296267625767470,8.92,8.92,"""25-0000401""",0.0,0.0,0
0,"""PAYMENT""",545.85,4178224023847746,93.6,-452.25,"""13-0001587""",0.0,545.85,0
0,"""TRANSFER""",19847.01,4178224023847746,-452.25,-20299.26,"""4096920916696293""",4011.72,23858.74,1
0,"""PAYMENT""",546.89,4779013371563747,159148.76,158601.88,"""75-0003564""",0.0,546.89,0
0,"""TRANSFER""",16973.75,4779013371563747,158601.88,141628.12,"""4593247015015219""",54.33,17028.08,1
0,"""PAYMENT""",659.02,4479448076398077,261950.48,261291.46,"""11-0003866""",0.0,659.02,0
0,"""TRANSFER""",21103.99,4479448076398077,261291.46,240187.47,"""4649550475321678""",19.63,21123.63,1
0,"""PAYMENT""",648.89,4587942659105635,4808.94,4160.05,"""64-0008405""",0.0,648.89,0


Sampel EDA menggunakan **100,000 baris**.

In [ ]:
null_summary = (
    lazy_df
    .select([
        pl.col(col).null_count().alias(col)
        for col in columns
    ])
    .collect(engine="streaming")
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"]
    )
    .with_columns(
        (
            pl.col("null_count") / row_count * 100
        ).round(4).alias("null_percentage")
    )
    .sort("null_count", descending=True)
)

columns_with_null = null_summary.filter(
    pl.col("null_count") > 0
)

if columns_with_null.height == 0:
    display(Markdown("**Tidak ditemukan missing value.**"))
else:
    display(columns_with_null)

**Tidak ditemukan missing value.**

In [ ]:
sample_schema = eda_df.schema

numeric_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype.is_numeric()
]

string_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype == pl.String
]

print("Kolom numerik:", numeric_cols)
print("Kolom teks:", string_cols)



Kolom numerik: ['step', 'amount', 'initiator', 'oldBalInitiator', 'newBalInitiator', 'oldBalRecipient', 'newBalRecipient', 'isFraud']
Kolom teks: ['transactionType', 'recipient']


In [ ]:
if numeric_cols:
    numeric_summary = eda_df.select(
        numeric_cols
    ).describe()

    display(numeric_summary)
else:
    display(Markdown(
        "**Tidak ditemukan kolom numerik pada hasil parsing awal.**"
    ))

statistic,step,amount,initiator,oldBalInitiator,newBalInitiator,oldBalRecipient,newBalRecipient,isFraud
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",5.56527,56655.657033,4.5010e15,287459.142625,332396.509199,7835.952872,12129.0505,0.11091
"""std""",2.983908,80062.006032,2.9027e14,569433.351835,601407.309418,37099.060757,42482.275094,0.314022
"""min""",0.0,1.36,4.0001e15,-199997.11,-199997.11,-149882.02,-135727.98,0.0
"""25%""",3.0,598.6,4.2472e15,18.44,20.4,534.48,1159.69,0.0
"""50%""",7.0,17572.83,4.5055e15,37481.59,83645.61,1735.43,2377.61,0.0
"""75%""",8.0,91043.64,4.7535e15,340451.41,425667.37,3303.1,4703.7,0.0
"""max""",9.0,1308624.7,4.9999e15,5.5312e6,5662492.3,1.5432e6,2.6180e6,1.0


In [ ]:
if string_cols:
    unique_summary = pl.DataFrame({
        "column": string_cols,
        "unique_count_sample": [
            eda_df[col].n_unique()
            for col in string_cols
        ]
    }).sort(
        "unique_count_sample",
        descending=True
    )

    display(unique_summary)

column,unique_count_sample
str,i64
"""recipient""",10090
"""transactionType""",5


In [ ]:
duplicate_count_sample = int(
    eda_df.is_duplicated().sum()
)

duplicate_percentage = (
    duplicate_count_sample / eda_df.height * 100
)

display(Markdown(
    f"""
**Duplikasi identik pada sampel:**
- Jumlah: **{duplicate_count_sample:,} baris**
- Persentase: **{duplicate_percentage:.4f}%**
"""
))


**Duplikasi identik pada sampel:**  
- Jumlah: **0 baris**
- Persentase: **0.0000%**


In [ ]:
target_keywords = [
    "fraud",
    "label",
    "class",
    "target",
    "anomaly"
]

time_keywords = [
    "date",
    "time",
    "timestamp",
    "step"
]

target_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in target_keywords)
]

time_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in time_keywords)
]

display(Markdown(
    f"""
### Kandidat Kolom Penting

**Kandidat label:** `{target_candidates}`

**Kandidat waktu:** `{time_candidates}`
"""
))


### Kandidat Kolom Penting

**Kandidat label:** `['isFraud']`

**Kandidat waktu:** `['step']`


In [ ]:
for target_col in target_candidates:
    target_distribution = (
        lazy_df
        .group_by(target_col)
        .agg(
            pl.len().alias("count")
        )
        .with_columns(
            (
                pl.col("count") / row_count * 100
            ).round(4).alias("percentage")
        )
        .sort("count", descending=True)
        .collect(engine="streaming")
    )

    display(Markdown(
        f"### Distribusi `{target_col}`"
    ))
    display(target_distribution)

### Distribusi `isFraud`

isFraud,count,percentage
i64,u32,f64
0,1544663,89.7965
1,175518,10.2035


In [ ]:
import re


def normalize_column_name(name: str) -> str:
    """Mengubah nama kolom menjadi format snake_case."""
    name = name.strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_").lower()


rename_map = {
    col: normalize_column_name(col)
    for col in columns
}

clean_lf = lazy_df.rename(rename_map)

display(
    pl.DataFrame({
        "original_column": list(rename_map.keys()),
        "clean_column": list(rename_map.values())
    })
)

original_column,clean_column
str,str
"""step""","""step"""
"""transactionType""","""transaction_type"""
"""amount""","""amount"""
"""initiator""","""initiator"""
"""oldBalInitiator""","""old_bal_initiator"""
"""newBalInitiator""","""new_bal_initiator"""
"""recipient""","""recipient"""
"""oldBalRecipient""","""old_bal_recipient"""
"""newBalRecipient""","""new_bal_recipient"""


In [ ]:
clean_schema = clean_lf.collect_schema()

clean_string_cols = [
    col
    for col, dtype in clean_schema.items()
    if dtype == pl.String
]

NULL_TOKENS = [
    "",
    "na",
    "n/a",
    "null",
    "none",
    "-"
]

clean_lf = clean_lf.with_columns([
    pl.when(
        pl.col(col)
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(NULL_TOKENS)
    )
    .then(None)
    .otherwise(
        pl.col(col).str.strip_chars()
    )
    .alias(col)
    for col in clean_string_cols
])

display(Markdown(
    f"Kolom teks yang dibersihkan: **{len(clean_string_cols)} kolom**."
))

Kolom teks yang dibersihkan: **2 kolom**.

In [ ]:
STAGE_1_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_stage1_clean.parquet"
)

clean_lf.sink_parquet(
    str(STAGE_1_PATH),
    compression="zstd"
)

display(Markdown(
    f"""
### Data Cleaning Tahap Awal Selesai

File tersimpan di:

`{STAGE_1_PATH}`
"""
))

clean_lf = pl.scan_parquet(
    STAGE_1_PATH
)


### Data Cleaning Tahap Awal Selesai

File tersimpan di:

`/content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_stage1_clean.parquet`


In [ ]:
import polars as pl
from IPython.display import display, Markdown

# Kolom utama berdasarkan dataset kamu
TIME_COL = "step"
TARGET_COL = "is_fraud"

# Pastikan kolom wajib tersedia
required_columns = [
    "step",
    "transaction_type",
    "amount",
    "initiator",
    "old_bal_initiator",
    "new_bal_initiator",
    "recipient",
    "old_bal_recipient",
    "new_bal_recipient",
    "is_fraud",
]

available_columns = clean_lf.collect_schema().names()

missing_columns = [
    col for col in required_columns
    if col not in available_columns
]

if missing_columns:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_columns}"
    )

print("Semua kolom wajib tersedia.")

Semua kolom wajib tersedia.


In [ ]:
# ============================================================
# FEATURE ENGINEERING AWAL
# ============================================================

feature_lf = (
    clean_lf
    .with_columns([
        # Tipe data dibuat konsisten
        pl.col("step").cast(pl.Int64),
        pl.col("amount").cast(pl.Float64),
        pl.col("initiator").cast(pl.String),
        pl.col("recipient").cast(pl.String),
        pl.col("is_fraud").cast(pl.Int8).alias("target"),

        pl.col("transaction_type")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase(),
    ])
    .filter(
        pl.col("step").is_not_null() &
        pl.col("amount").is_not_null() &
        pl.col("target").is_not_null()
    )
    .with_columns([
        # Mengurangi efek nominal yang sangat ekstrem
        pl.col("amount")
        .clip(lower_bound=0)
        .log1p()
        .alias("amount_log1p"),

        # Perubahan saldo aktual
        (
            pl.col("new_bal_initiator") -
            pl.col("old_bal_initiator")
        ).alias("initiator_balance_change"),

        (
            pl.col("new_bal_recipient") -
            pl.col("old_bal_recipient")
        ).alias("recipient_balance_change"),

        # Rasio nominal terhadap saldo awal
        (
            pl.col("amount") /
            (pl.col("old_bal_initiator").abs() + 1)
        ).alias("amount_to_initiator_balance_ratio"),

        (
            pl.col("amount") /
            (pl.col("old_bal_recipient").abs() + 1)
        ).alias("amount_to_recipient_balance_ratio"),

        # Seberapa besar saldo pengirim berkurang
        (
            (
                pl.col("old_bal_initiator") -
                pl.col("new_bal_initiator")
            )
            / (pl.col("old_bal_initiator").abs() + 1)
        ).alias("initiator_balance_drop_ratio"),

        # Penanda kondisi saldo tertentu
        (
            pl.col("old_bal_initiator") <= 0
        ).cast(pl.Int8).alias("initiator_old_balance_non_positive"),

        (
            pl.col("new_bal_initiator") < 0
        ).cast(pl.Int8).alias("initiator_new_balance_negative"),

        (
            pl.col("old_bal_recipient") == 0
        ).cast(pl.Int8).alias("recipient_old_balance_zero"),
    ])
)

In [ ]:
feature_lf = feature_lf.with_columns([
    # Saldo pengirim yang secara sederhana diharapkan
    (
        pl.col("old_bal_initiator") -
        pl.col("amount")
    ).alias("expected_new_bal_initiator"),

    # Saldo penerima yang secara sederhana diharapkan
    (
        pl.col("old_bal_recipient") +
        pl.col("amount")
    ).alias("expected_new_bal_recipient"),
])

feature_lf = feature_lf.with_columns([
    # Selisih antara saldo aktual dan saldo yang diharapkan
    (
        pl.col("new_bal_initiator") -
        pl.col("expected_new_bal_initiator")
    ).abs().alias("initiator_balance_error"),

    (
        pl.col("new_bal_recipient") -
        pl.col("expected_new_bal_recipient")
    ).abs().alias("recipient_balance_error"),
])

In [ ]:
import numpy as np
import polars as pl

feature_lf = feature_lf.with_columns([
    # Jam relatif dalam sehari: 0 sampai 23
    (pl.col("step") % 24)
    .cast(pl.Int8)
    .alias("hour_of_day"),

    # Indeks hari sejak simulasi dimulai: 0 sampai 29
    (pl.col("step") // 24)
    .cast(pl.Int16)
    .alias("day_index"),
])

feature_lf = feature_lf.with_columns([
    # Transaksi malam, sementara didefinisikan pukul 22.00 sampai 04.59
    (
        (pl.col("hour_of_day") >= 22) |
        (pl.col("hour_of_day") < 5)
    )
    .cast(pl.Int8)
    .alias("is_night"),

    # Jam bersifat siklik. Jam 23 dekat dengan jam 0.
    (
        2 * np.pi * pl.col("hour_of_day") / 24
    )
    .sin()
    .alias("hour_sin"),

    (
        2 * np.pi * pl.col("hour_of_day") / 24
    )
    .cos()
    .alias("hour_cos"),
])

In [ ]:
# ============================================================
# TEMPORAL SPLIT BERDASARKAN STEP
# ============================================================

unique_steps = (
    feature_lf
    .select("step")
    .unique()
    .sort("step")
    .collect(engine="streaming")
    .get_column("step")
)

n_steps = len(unique_steps)

train_step_index = int(n_steps * 0.70) - 1
validation_step_index = int(n_steps * 0.85) - 1

train_end_step = unique_steps[train_step_index]
validation_end_step = unique_steps[validation_step_index]

print("Jumlah step unik:", n_steps)
print("Train sampai step:", train_end_step)
print("Validation sampai step:", validation_end_step)
print("Test setelah step:", validation_end_step)

Jumlah step unik: 144
Train sampai step: 99
Validation sampai step: 121
Test setelah step: 121


In [ ]:
train_lf = feature_lf.filter(
    pl.col("step") <= train_end_step
)

validation_lf = feature_lf.filter(
    (pl.col("step") > train_end_step) &
    (pl.col("step") <= validation_end_step)
)

test_lf = feature_lf.filter(
    pl.col("step") > validation_end_step
)

In [ ]:
def summarize_split(
    split_name: str,
    split_lf: pl.LazyFrame
) -> pl.DataFrame:
    return (
        split_lf
        .select([
            pl.lit(split_name).alias("split"),
            pl.len().alias("row_count"),
            pl.col("step").min().alias("step_start"),
            pl.col("step").max().alias("step_end"),
            pl.col("target").sum().alias("fraud_count"),
            pl.col("target").mean().alias("fraud_rate"),
        ])
        .collect(engine="streaming")
    )


split_summary = pl.concat([
    summarize_split("train", train_lf),
    summarize_split("validation", validation_lf),
    summarize_split("test", test_lf),
])

display(split_summary)

split,row_count,step_start,step_end,fraud_count,fraud_rate
str,u32,i64,i64,i64,f64
"""train""",1242432,0,99,122258,0.098402
"""validation""",183714,100,121,26545,0.144491
"""test""",294035,122,143,26715,0.090857


In [ ]:
TRAIN_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_train_features_v1.parquet"
)

VALIDATION_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_validation_features_v1.parquet"
)

TEST_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_test_features_v1.parquet"
)

train_lf.sink_parquet(
    str(TRAIN_PATH),
    compression="zstd"
)

validation_lf.sink_parquet(
    str(VALIDATION_PATH),
    compression="zstd"
)

test_lf.sink_parquet(
    str(TEST_PATH),
    compression="zstd"
)

print("Train:", TRAIN_PATH)
print("Validation:", VALIDATION_PATH)
print("Test:", TEST_PATH)

Train: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_train_features_v1.parquet
Validation: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_validation_features_v1.parquet
Test: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_test_features_v1.parquet


In [ ]:
BASELINE_NUMERIC_FEATURES = [
    "amount",
    "amount_log1p",
    "old_bal_initiator",
    "new_bal_initiator",
    "old_bal_recipient",
    "new_bal_recipient",
    "initiator_balance_change",
    "recipient_balance_change",
    "amount_to_initiator_balance_ratio",
    "amount_to_recipient_balance_ratio",
    "initiator_balance_drop_ratio",
    "initiator_old_balance_non_positive",
    "initiator_new_balance_negative",
    "recipient_old_balance_zero",
    "initiator_balance_error",
    "recipient_balance_error",
    "hour_of_day",
    "is_night",
    "hour_sin",
    "hour_cos",
]

BASELINE_CATEGORICAL_FEATURES = [
    "transaction_type",
]

TARGET = "target"

print("Numeric features:", len(BASELINE_NUMERIC_FEATURES))
print("Categorical features:", BASELINE_CATEGORICAL_FEATURES)

Numeric features: 20
Categorical features: ['transaction_type']


In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation,
)

SEED = 42

In [ ]:
NUMERIC_FEATURES = [
    "amount",
    "amount_log1p",
    "old_bal_initiator",
    "new_bal_initiator",
    "old_bal_recipient",
    "new_bal_recipient",
    "initiator_balance_change",
    "recipient_balance_change",
    "amount_to_initiator_balance_ratio",
    "amount_to_recipient_balance_ratio",
    "initiator_balance_drop_ratio",
    "initiator_old_balance_non_positive",
    "initiator_new_balance_negative",
    "recipient_old_balance_zero",

    # Fitur waktu
    "hour_of_day",
    "is_night",
    "hour_sin",
    "hour_cos",
]

CATEGORICAL_FEATURES = [
    "transaction_type",
]

TARGET = "target"

MODEL_COLUMNS = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
    + [TARGET]
)

In [ ]:
def load_model_split(path):
    numeric_expressions = [
        pl.col(column)
        .cast(pl.Float32, strict=False)
        .fill_nan(0)
        .fill_null(0)
        .alias(column)
        for column in NUMERIC_FEATURES
    ]

    categorical_expressions = [
        pl.col(column)
        .cast(pl.String)
        .fill_null("UNKNOWN")
        .alias(column)
        for column in CATEGORICAL_FEATURES
    ]

    return (
        pl.scan_parquet(str(path))
        .select(MODEL_COLUMNS)
        .with_columns(
            numeric_expressions
            + categorical_expressions
            + [
                pl.col(TARGET)
                .cast(pl.Int8)
                .alias(TARGET)
            ]
        )
        .collect(engine="streaming")
        .to_pandas()
    )


train_df = load_model_split(TRAIN_PATH)
validation_df = load_model_split(VALIDATION_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

print(
    "Train memory:",
    round(
        train_df.memory_usage(deep=True).sum() / 1024**2,
        2
    ),
    "MB"
)

Train shape: (1242432, 20)
Validation shape: (183714, 20)
Train memory: 153.42 MB


In [ ]:
X_train = train_df[
    NUMERIC_FEATURES + CATEGORICAL_FEATURES
]

y_train = train_df[TARGET].astype("int8")

X_validation = validation_df[
    NUMERIC_FEATURES + CATEGORICAL_FEATURES
]

y_validation = validation_df[TARGET].astype("int8")

print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_validation.mean())

Train fraud rate: 0.09840216607428012
Validation fraud rate: 0.1444908934539556


In [ ]:
def find_best_f1_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        probabilities
    )

    # Precision dan recall memiliki satu elemen lebih banyak
    # dibanding daftar threshold.
    precision = precision[:-1]
    recall = recall[:-1]

    denominator = precision + recall

    f1_values = np.divide(
        2 * precision * recall,
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0,
    )

    best_index = int(np.argmax(f1_values))

    return {
        "threshold": float(thresholds[best_index]),
        "precision": float(precision[best_index]),
        "recall": float(recall[best_index]),
        "f1": float(f1_values[best_index]),
    }


def evaluate_probabilities(
    model_name,
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": model_name,
        "threshold": round(float(threshold), 6),
        "pr_auc": round(
            average_precision_score(
                y_true,
                probabilities
            ),
            6
        ),
        "precision": round(
            precision_score(
                y_true,
                predictions,
                zero_division=0
            ),
            6
        ),
        "recall": round(
            recall_score(
                y_true,
                predictions,
                zero_division=0
            ),
            6
        ),
        "f1": round(
            f1_score(
                y_true,
                predictions,
                zero_division=0
            ),
            6
        ),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_negative": int(tn),
        "false_positive_rate": round(
            fp / max(fp + tn, 1),
            6
        ),
    }

# LOGISTIC REGRESSION

In [ ]:
logreg_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

logreg_model = Pipeline([
    (
        "preprocessor",
        logreg_preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            solver="saga",
            class_weight="balanced",
            max_iter=300,
            tol=1e-3,
            random_state=SEED,
        )
    ),
])

In [ ]:
%%time

logreg_model.fit(
    X_train,
    y_train
)

CPU times: user 5min 58s, sys: 1.05 s, total: 6min
Wall time: 6min 22s


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(sparse_threshold=1.0,
                                   transformers=[('numeric', StandardScaler(),
                                                  ['amount', 'amount_log1p',
                                                   'old_bal_initiator',
                                                   'new_bal_initiator',
                                                   'old_bal_recipient',
                                                   'new_bal_recipient',
                                                   'initiator_balance_change',
                                                   'recipient_balance_change',
                                                   'amount_to_initiator_balance_ratio',
                                                   'amount_to_recipient_balance_ratio',
                                                   'initi...op_ratio',
                                                   'initiator_old_balance_non_positive',
                                                   'initiator_new_balance_negative',
                                                   'recipient_old_balance_zero',
                                                   'hour_of_day', 'is_night',
                                                   'hour_sin', 'hour_cos']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['transaction_type'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=300,
                                    random_state=42, solver='saga',
                                    tol=0.001))])

In [ ]:
logreg_validation_probabilities = (
    logreg_model.predict_proba(
        X_validation
    )[:, 1]
)

logreg_best = find_best_f1_threshold(
    y_validation,
    logreg_validation_probabilities
)

print("Best threshold Logistic Regression:")
print(logreg_best)

Best threshold Logistic Regression:
{'threshold': 0.6040356428448024, 'precision': 0.30766466093932965, 'recall': 0.9996986249764551, 'f1': 0.47052252699515945}


In [ ]:
logreg_default_metrics = evaluate_probabilities(
    model_name="Logistic Regression - threshold 0.5",
    y_true=y_validation,
    probabilities=logreg_validation_probabilities,
    threshold=0.5,
)

logreg_tuned_metrics = evaluate_probabilities(
    model_name="Logistic Regression - tuned",
    y_true=y_validation,
    probabilities=logreg_validation_probabilities,
    threshold=logreg_best["threshold"],
)

pd.DataFrame([
    logreg_default_metrics,
    logreg_tuned_metrics,
])

,model,threshold,pr_auc,precision,recall,f1,true_positive,false_positive,false_negative,true_negative,false_positive_rate
0,Logistic Regression - threshold 0.5,0.500000,0.330653,0.307568,0.999812,0.470421,26540,59750,5,97419,0.380164
1,Logistic Regression - tuned,0.604036,0.330653,0.307665,0.999699,0.470523,26537,59716,8,97453,0.379948


# LightGBM

In [ ]:
transaction_type_categories = sorted(
    X_train["transaction_type"]
    .dropna()
    .unique()
    .tolist()
)

transaction_type_dtype = pd.CategoricalDtype(
    categories=transaction_type_categories
)


def prepare_lightgbm_data(X):
    prepared = X.copy()

    prepared["transaction_type"] = (
        prepared["transaction_type"]
        .astype(transaction_type_dtype)
    )

    return prepared


X_train_lgbm = prepare_lightgbm_data(X_train)
X_validation_lgbm = prepare_lightgbm_data(X_validation)

In [ ]:
lightgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

In [ ]:
%%time

lightgbm_model.fit(
    X_train_lgbm,
    y_train,
    eval_set=[
        (
            X_validation_lgbm,
            y_validation
        )
    ],
    eval_metric="average_precision",
    categorical_feature=[
        "transaction_type"
    ],
    callbacks=[
        early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True,
        ),
        log_evaluation(period=50),
    ],
)

[LightGBM] [Info] Number of positive: 122258, number of negative: 1120174
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.184285 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2871
[LightGBM] [Info] Number of data points in the train set: 1242432, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.340776	valid_0's binary_logloss: 0.551202
[100]	valid_0's average_precision: 0.341507	valid_0's binary_logloss: 0.562676
Early stopping, best iteration is:
[91]	valid_0's average_precision: 0.342034	valid_0's binary_logloss: 0.561873
Evaluated only: average_precision
CPU times: user 25.6 s, sys: 117 ms, total: 25.8 s
Wall time: 14.4 s


LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
               learning_rate=0.05, min_child_samples=50, n_estimators=1000,
               n_jobs=-1, objective='binary', random_state=42, reg_alpha=0.1,
               reg_lambda=0.1, subsample=0.8)

In [ ]:
lightgbm_validation_probabilities = (
    lightgbm_model.predict_proba(
        X_validation_lgbm,
        num_iteration=lightgbm_model.best_iteration_,
    )[:, 1]
)

lightgbm_best = find_best_f1_threshold(
    y_validation,
    lightgbm_validation_probabilities
)

print("Best iteration:", lightgbm_model.best_iteration_)
print("Best threshold LightGBM:")
print(lightgbm_best)

Best iteration: 91
Best threshold LightGBM:
{'threshold': 0.7099354482411558, 'precision': 0.30767446445761476, 'recall': 0.999359578074967, 'f1': 0.4704964262277637}


In [ ]:
lightgbm_default_metrics = evaluate_probabilities(
    model_name="LightGBM - threshold 0.5",
    y_true=y_validation,
    probabilities=lightgbm_validation_probabilities,
    threshold=0.5,
)

lightgbm_tuned_metrics = evaluate_probabilities(
    model_name="LightGBM - tuned",
    y_true=y_validation,
    probabilities=lightgbm_validation_probabilities,
    threshold=lightgbm_best["threshold"],
)

pd.DataFrame([
    lightgbm_default_metrics,
    lightgbm_tuned_metrics,
])

,model,threshold,pr_auc,precision,recall,f1,true_positive,false_positive,false_negative,true_negative,false_positive_rate
0,LightGBM - threshold 0.5,0.500000,0.342034,0.307592,0.999925,0.470462,26543,59750,2,97419,0.380164
1,LightGBM - tuned,0.709935,0.342034,0.307674,0.999360,0.470496,26528,59693,17,97476,0.379801


# COMPARISON 2 MODEL

In [ ]:
model_comparison = (
    pd.DataFrame([
        logreg_tuned_metrics,
        lightgbm_tuned_metrics,
    ])
    .sort_values(
        by="pr_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

display(model_comparison)

,model,threshold,pr_auc,precision,recall,f1,true_positive,false_positive,false_negative,true_negative,false_positive_rate
0,LightGBM - tuned,0.709935,0.342034,0.307674,0.999360,0.470496,26528,59693,17,97476,0.379801
1,Logistic Regression - tuned,0.604036,0.330653,0.307665,0.999699,0.470523,26537,59716,8,97453,0.379948


# SIMPAN MODEL BASELINE

In [ ]:
joblib.dump(
    logreg_model,
    MODEL_DIR / "logistic_regression_baseline_v1.joblib"
)

lightgbm_model.booster_.save_model(
    str(
        MODEL_DIR /
        "lightgbm_baseline_v1.txt"
    )
)

print("Baseline models berhasil disimpan.")